# Set Piece H2H Tinker Notebook

Use this notebook to pull backend set piece data for one opposition club and iterate on the chart design locally.

Typical workflow:
1. Run Cell 2 to connect to DuckDB.
2. Edit parameters in Cell 3 (`set_piece`, `opposition_club`, filters).
3. Run Cells 4-5 to rebuild data and redraw charts.

In [12]:
from pathlib import Path
import sys

import altair as alt
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'python').exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from python.backend import BackendDatabase

from python.charts import alt_theme

db = BackendDatabase()
alt.data_transformers.disable_max_rows()
alt.themes.register('alt_theme', alt_theme)
alt.themes.enable('alt_theme')

print(f'Connected to backend: {db.db_file}')

Connected to backend: /Users/samlindsay/Documents/Projects/Personal/EGRFC/egrfc-stats/data/egrfc_backend.duckdb


In [13]:
set_piece = 'Lineout'  # 'Lineout' or 'Scrum'
opposition_club = 'Haywards Heath'

squad_filter = 'All'      # 'All', '1st', '2nd'

assert set_piece in {'Lineout', 'Scrum'}

In [14]:
metric_won = 'lineouts_won' if set_piece.lower() == 'lineout' else 'scrums_won'
metric_total = 'lineouts_total' if set_piece.lower() == 'lineout' else 'scrums_total'

sql = f'''
SELECT
    G.game_id,
    G.date,
    G.squad,
    G.season,
    G.game_type,
    G.opposition,
    G.home_away,
    CASE WHEN S.team = 'EGRFC' THEN 'EGRFC' ELSE 'Opposition' END AS team,
    S.{metric_won}::DOUBLE AS won,
    (S.{metric_total} - S.{metric_won})::DOUBLE AS lost,
    S.{metric_total}::DOUBLE AS total
FROM set_piece S
JOIN games G USING (game_id)
WHERE S.{metric_total} > 0
ORDER BY G.date DESC
'''
raw = db.con.execute(sql).df()

raw['season'] = raw['season'].astype(str)
raw['game_type'] = raw['game_type'].fillna('Unknown').astype(str)
raw['opposition'] = raw['opposition'].fillna('Unknown').astype(str)
raw['opposition_club'] = (
    raw['opposition']
    .str.replace(r'\s+(?:I{1,6}|[1-6](?:st|nd|rd|th)?|A|B)(?:\s+XV)?$', '', regex=True)
    .str.strip()
)
raw.loc[raw['opposition_club'] == '', 'opposition_club'] = 'Unknown'

df = raw[raw['opposition_club'] == opposition_club].copy()
if squad_filter != 'All':
    df = df[df['squad'].astype(str) == str(squad_filter)]

if df.empty:
    raise ValueError('No rows for the chosen opposition/filter combination.')

df['won'] = df['won'].clip(lower=0)
df['total'] = df[['total', 'won']].max(axis=1)
df['lost'] = (df['total'] - df['won']).clip(lower=0)
df['game_axis_key'] = df.apply(lambda r: f"{r['game_id']}__{r['squad']} XV v {r['opposition']} ({r['home_away']})", axis=1)
df['game_label'] = df['opposition'] + ' (' + df['home_away'].astype(str) + ')'
df['team_label'] = df['team'].map({'EGRFC': 'East Grinstead', 'Opposition': 'Opposition'})

fixture_cols = ['date', 'season', 'squad', 'game_type', 'opposition', 'team', 'won', 'lost', 'total']
display(df.sort_values(['date', 'team']).reset_index(drop=True)[fixture_cols])
print(f'Rows: {len(df)} | Games: {df.game_id.nunique()}')

,date,season,squad,game_type,opposition,team,won,lost,total
0,2022-11-26,2022/23,1st,League,Haywards Heath,EGRFC,9.0,4.0,13.0
1,2022-11-26,2022/23,1st,League,Haywards Heath,Opposition,7.0,4.0,11.0
2,2023-03-04,2022/23,1st,League,Haywards Heath,EGRFC,8.0,4.0,12.0
3,2023-03-04,2022/23,1st,League,Haywards Heath,Opposition,12.0,2.0,14.0
4,2024-09-21,2024/25,1st,League,Haywards Heath,EGRFC,7.0,2.0,9.0
5,2024-09-21,2024/25,1st,League,Haywards Heath,Opposition,10.0,4.0,14.0
6,2025-04-12,2024/25,1st,Cup,Haywards Heath,EGRFC,10.0,1.0,11.0
7,2025-04-12,2024/25,1st,Cup,Haywards Heath,Opposition,2.0,7.0,9.0


Rows: 8 | Games: 4


In [15]:
success_df = df.copy()
success_df['success_rate'] = success_df.apply(lambda r: (r['won'] / r['total']) if r['total'] else 0.0, axis=1)

def ordinal_date_label(value):
    d = pd.to_datetime(value)
    day = int(d.day)
    if 11 <= (day % 100) <= 13:
        suffix = 'th'
    else:
        suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(day % 10, 'th')
    return f"{day}{suffix} {d.strftime('%b %Y')}"

success_df['fixture_label'] = success_df['game_axis_key'].str.split('__').str[1]
success_df['date_label'] = success_df['date'].apply(ordinal_date_label)
success_df['y_axis'] = success_df['date_label'] + '||' + success_df['fixture_label']
success_df['put_in'] = success_df['team']  # Put-in is the team with the set piece

segment_rows = []
for row in success_df.itertuples(index=False):
    segments = [
        ('Retained', row.won if row.team == 'EGRFC' else -row.won, row.won),
        ('Turnover', -row.lost if row.team == 'EGRFC' else row.lost, row.lost),
    ]
    for outcome, signed_value, count in segments:
        segment_rows.append({
            'game_id': row.game_id,
            'date': row.date,
            'game_axis_key': row.game_axis_key,
            'game_label': row.game_label,
            'team': row.team,
            'team_label': row.team_label,
            'put_in': row.put_in,
            'outcome': outcome,
            'count': count,
            'segment_start': 0.0,
            'segment_end': signed_value,
        })

segment_df = pd.DataFrame(segment_rows).merge(
    success_df[['game_id', 'y_axis']].drop_duplicates(),
    on='game_id',
    how='left'
)

connector_df = (
    success_df.pivot_table(
        index=['game_id', 'game_axis_key', 'game_label', 'date', 'y_axis'],
        columns='team',
        values='success_rate',
        aggfunc='mean',
    )
    .reset_index()
    .rename_axis(None, axis=1)
)
connector_df['eg_rate'] = connector_df.get('EGRFC', pd.Series(index=connector_df.index, dtype=float)).fillna(0.0)
connector_df['opp_rate'] = connector_df.get('Opposition', pd.Series(index=connector_df.index, dtype=float)).fillna(0.0)
connector_df['success_diff'] = connector_df['eg_rate'] - connector_df['opp_rate']
connector_df['winner'] = connector_df['success_diff'].apply(lambda d: 'EGRFC' if d > 0 else ('Opposition' if d < 0 else 'Level'))

# Add put_in info to connector_df
connector_df = connector_df.merge(
    success_df[['game_id', 'put_in']].drop_duplicates(),
    on='game_id',
    how='left'
)

success_df = success_df.merge(connector_df[['game_id', 'eg_rate', 'opp_rate']], on='game_id', how='left')
success_df['is_lower'] = (
    ((success_df['team'] == 'EGRFC') & (success_df['eg_rate'] < success_df['opp_rate']))
    | ((success_df['team'] == 'Opposition') & (success_df['opp_rate'] < success_df['eg_rate']))
)

aggregate_segments = (
    segment_df.groupby(['team', 'team_label', 'outcome'], as_index=False)['count']
    .mean()
    .assign(
        game_axis_key='aggregate__AVERAGE',
        game_label='AVERAGE',
        y_axis='AVERAGE||AVERAGE',
        segment_start=0.0,
        segment_end=lambda x: x.apply(
            lambda r: r['count'] if (r['team'] == 'Opposition' and r['outcome'] == 'Turnover') or (r['team'] == 'EGRFC' and r['outcome'] == 'Retained') else -r['count'],
            axis=1
        ),
    )
)

aggregate_success = (
    success_df.groupby(['team', 'team_label'], as_index=False)
    .agg(
        won=('won', 'mean'),
        lost=('lost', 'mean'),
        total=('total', 'mean'),
        success_rate=('success_rate', 'mean'),
        put_in=('put_in', 'first'),
    )
)
aggregate_success['game_axis_key'] = 'aggregate__AVERAGE'
aggregate_success['y_axis'] = 'AVERAGE||AVERAGE'

eg_rate = float(aggregate_success.loc[aggregate_success['team'] == 'EGRFC', 'success_rate'].iloc[0]) if not aggregate_success.loc[aggregate_success['team'] == 'EGRFC'].empty else 0.0
opp_rate = float(aggregate_success.loc[aggregate_success['team'] == 'Opposition', 'success_rate'].iloc[0]) if not aggregate_success.loc[aggregate_success['team'] == 'Opposition'].empty else 0.0
aggregate_connector = pd.DataFrame([{'game_axis_key': 'aggregate__AVERAGE', 'game_label': 'AVERAGE', 'y_axis': 'AVERAGE||AVERAGE', 'eg_rate': eg_rate, 'opp_rate': opp_rate, 'success_diff': eg_rate - opp_rate}])
aggregate_connector['winner'] = aggregate_connector['success_diff'].apply(lambda d: 'EGRFC' if d > 0 else ('Opposition' if d < 0 else 'Level'))
aggregate_success['is_lower'] = (
    ((aggregate_success['team'] == 'EGRFC') & (eg_rate < opp_rate))
    | ((aggregate_success['team'] == 'Opposition') & (opp_rate < eg_rate))
)

count_max = max(1.0, float(segment_df['segment_end'].abs().max()), float(aggregate_segments['segment_end'].abs().max()))
count_domain = [-count_max, count_max]

# Scales
color_scale = alt.Scale(domain=['EGRFC', 'Opposition'], range=['#202946', '#991515'])
outcome_opacity_scale = alt.Scale(domain=['Retained', 'Turnover'], range=[0.5, 1.0])
put_in_scale = alt.Scale(domain=['EGRFC', 'Opposition'], range=['#202946', '#991515'])
winner_scale = alt.Scale(domain=['EGRFC', 'Opposition', 'Level'], range=['#202946', '#991515', '#666666'])
y_sort = alt.EncodingSortField(field='date', order='descending')

# Axis presets
count_axis_top = alt.Axis(
    format='d',
    labelExpr='abs(datum.value)',
    orient='top',
    labelPadding=15,
    titlePadding=8,
)
count_axis_bottom = alt.Axis(
    format='d',
    labelExpr='abs(datum.value)',
    orient='bottom',
)

# Background zone annotations — shared by both panels
zone_df = pd.DataFrame({
    'x1': [-count_max, 0.0],
    'x2': [0.0, count_max],
    'x_mid': [-count_max / 2, count_max / 2],
    'label': ['← Opposition wins', 'East Grinstead wins →'],
    'fill': ['#991515', '#202946'],
})

bg_rects = alt.Chart(zone_df).mark_rect(opacity=0.04).encode(
    x=alt.X('x1:Q', scale=alt.Scale(domain=count_domain)),
    x2='x2:Q',
    color=alt.Color('fill:N', scale=None, legend=None),
)

bg_text = alt.Chart(zone_df).mark_text(
    align='center', baseline='middle', fontSize=12, fontWeight='bold', fontStyle='italic', opacity=0.4,
).encode(
    x=alt.X('x_mid:Q', scale=alt.Scale(domain=count_domain)),
    y=alt.value(-15),  # Position above the top axis
    text='label:N',
    color=alt.Color('fill:N', scale=None, legend=None),
)

# --- Per-game panel ---
flow = alt.Chart(segment_df).mark_bar(stroke='white', strokeWidth=0.8, size=16).encode(
    y=alt.Y(
        'y_axis:N',
        sort=y_sort,
        title=None,
        axis=alt.Axis(labelLimit=220, ticks=False, domain=False, labelExpr="split(datum.label, '||')[1]", labelPadding=10),
        scale=alt.Scale()
    ),
    yOffset=alt.YOffset('team:N', sort=['Opposition', 'EGRFC'], scale=alt.Scale()),
    x=alt.X('segment_start:Q', title=f'{set_piece}s', scale=alt.Scale(domain=count_domain), axis=count_axis_top),
    x2='segment_end:Q',
    color=alt.Color('team:N', scale=color_scale,
        legend=alt.Legend(title='Put-in', orient='bottom', direction='horizontal', columns=2)
    ),
    opacity=alt.Opacity('outcome:N', scale=outcome_opacity_scale,
        legend=alt.Legend(title='Outcome', orient='bottom', direction='horizontal', columns=2)
    ),
    tooltip=['game_label:N', 'team_label:N', 'outcome:N', alt.Tooltip('count:Q', title='Count', format=',.1f')]
).properties(width=320, height=alt.Step(18))

success_connectors = alt.Chart(connector_df).mark_rule(strokeWidth=3, opacity=0.6).encode(
    y=alt.Y(
        'y_axis:N',
        sort=y_sort,
        title=None,
        axis=alt.Axis(orient='right', labelLimit=220, ticks=False, domain=False, labelExpr="split(datum.label, '||')[0]", labelPadding=10),
        scale=alt.Scale()
    ),
    x=alt.X('eg_rate:Q', scale=alt.Scale(domain=[0, 1]), axis=alt.Axis(format='%', orient='top')),
    x2='opp_rate:Q',
    color=alt.Color('winner:N', legend=None, scale=winner_scale),
    tooltip=['game_label:N', alt.Tooltip('eg_rate:Q', title='EGRFC Success', format='.1%'), alt.Tooltip('opp_rate:Q', title='Opposition Success', format='.1%')]
).properties(width=200, height=alt.Step(18))

success_points_higher = alt.Chart(success_df).transform_filter(
    "datum.is_lower == false"
).mark_point(size=170, filled=True, stroke='white', strokeWidth=1.4, opacity=1).encode(
    y=alt.Y('y_axis:N', sort=y_sort, title=None, axis=alt.Axis(orient='right')),
    x=alt.X('success_rate:Q', title="Success Rate", scale=alt.Scale(domain=[0, 1]), axis=alt.Axis(format='%', orient='top')),
    color=alt.Color('team:N', legend=None, scale=color_scale),
    tooltip=['game_label:N', 'team_label:N', alt.Tooltip('won:Q', title='Won', format=',.0f'), alt.Tooltip('lost:Q', title='Lost', format=',.0f'), alt.Tooltip('success_rate:Q', title='Success', format='.1%')]
)

success_points_lower = alt.Chart(success_df).transform_filter(
    "datum.is_lower == true"
).mark_point(size=170, filled=False, strokeWidth=2, opacity=1).encode(
    y=alt.Y('y_axis:N', sort=y_sort, title=None, axis=alt.Axis(orient='right')),
    x=alt.X('success_rate:Q', scale=alt.Scale(domain=[0, 1]), axis=alt.Axis(format='%', orient='top')),
    stroke=alt.Stroke('team:N', legend=None, scale=color_scale),
    tooltip=['game_label:N', 'team_label:N', alt.Tooltip('won:Q', title='Won', format=',.0f'), alt.Tooltip('lost:Q', title='Lost', format=',.0f'), alt.Tooltip('success_rate:Q', title='Success', format='.1%')]
)

main = alt.hconcat(
    bg_rects + flow,
    success_connectors + success_points_higher + success_points_lower,
    spacing=10
).resolve_scale(y='shared').properties(
    title=alt.TitleParams(text=f'{set_piece}s Head-to-Head', subtitle='Per-game mirrored bars plus success-rate panel')
)

# --- Average summary panel (legends suppressed to avoid duplication) ---
agg_flow = alt.Chart(aggregate_segments).mark_bar(stroke='white', strokeWidth=0.8, size=16).encode(
    y=alt.Y(
        'y_axis:N',
        title=None,
        axis=alt.Axis(labelExpr="split(datum.label, '||')[1]", labelFontWeight='bold', ticks=False, domain=False, labelPadding=10),
        scale=alt.Scale()
    ),
    yOffset=alt.YOffset('team:N', sort=['Opposition', 'EGRFC'], scale=alt.Scale()),
    x=alt.X('segment_start:Q', title=f'{set_piece}s', scale=alt.Scale(domain=count_domain), axis=count_axis_bottom),
    x2='segment_end:Q',
    color=alt.Color('team:N', scale=color_scale, legend=None),
    opacity=alt.Opacity('outcome:N', scale=outcome_opacity_scale, legend=None),
    tooltip=['team_label:N', 'outcome:N', alt.Tooltip('count:Q', title='Average Count', format=',.1f')]
).properties(width=320, height=alt.Step(18))

agg_connector = alt.Chart(aggregate_connector).mark_rule(strokeWidth=3, opacity=0.6).encode(
    y=alt.Y('y_axis:N', title=None, axis=alt.Axis(orient='right', labelExpr="split(datum.label, '||')[0]", labelFontWeight='bold', ticks=False, domain=False, labelPadding=10)),
    x=alt.X('eg_rate:Q', title="Success Rate", scale=alt.Scale(domain=[0, 1]), axis=alt.Axis(format='%', orient='bottom')),
    x2='opp_rate:Q',
    color=alt.Color('winner:N', legend=None, scale=winner_scale),
    tooltip=['game_label:N', alt.Tooltip('eg_rate:Q', title='EGRFC Success', format='.1%'), alt.Tooltip('opp_rate:Q', title='Opposition Success', format='.1%')]
).properties(width=200, height=alt.Step(18))

agg_points_higher = alt.Chart(aggregate_success).transform_filter(
    "datum.is_lower == false"
).mark_point(size=180, filled=True, stroke='white', strokeWidth=1.4, opacity=1).encode(
    y=alt.Y('y_axis:N', title=None, axis=alt.Axis(orient='right', labelFontWeight='bold')),
    x=alt.X('success_rate:Q', scale=alt.Scale(domain=[0, 1])),
    color=alt.Color('team:N', legend=None, scale=color_scale),
    tooltip=['team_label:N', alt.Tooltip('won:Q', title='Avg Won', format=',.1f'), alt.Tooltip('lost:Q', title='Avg Lost', format=',.1f'), alt.Tooltip('success_rate:Q', title='Avg Success', format='.1%')]
)

agg_points_lower = alt.Chart(aggregate_success).transform_filter(
    "datum.is_lower == true"
).mark_point(size=180, filled=False, strokeWidth=2, opacity=1).encode(
    y=alt.Y('y_axis:N', title=None, axis=alt.Axis(orient='right', labelFontWeight='bold')),
    x=alt.X('success_rate:Q', scale=alt.Scale(domain=[0, 1])),
    stroke=alt.Stroke('team:N', legend=None, scale=color_scale),
    tooltip=['team_label:N', alt.Tooltip('won:Q', title='Avg Won', format=',.1f'), alt.Tooltip('lost:Q', title='Avg Lost', format=',.1f'), alt.Tooltip('success_rate:Q', title='Avg Success', format='.1%')]
)

aggregate = alt.hconcat(
    bg_rects + bg_text + agg_flow,
    agg_connector + agg_points_higher + agg_points_lower,
    spacing=10
).resolve_scale(y='shared')

chart = alt.vconcat(main, aggregate, spacing=10)
chart

alt.VConcatChart(...)

In [ ]:
# Optional cleanup when finished
# db.close()